In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()

In [2]:
def create_8bit_adder_subtractor(a, b, mode):
    """
    8-bit reversible-style arithmetic circuit.

    a    : integer 0-255
    b    : integer 0-255
    mode : 0 = addition, 1 = subtraction

    Qubits:
        q0-q7   -> A bits
        q8-q15  -> B bits
        q16     -> Mode
        q17-q24 -> Result bits
        q25     -> Carry/Borrow
    """

    qc = QuantumCircuit(26, 26)

    # -----------------------------------------
    # Prepare A
    # -----------------------------------------

    for i in range(8):
        if (a >> i) & 1:
            qc.x(i)

    # -----------------------------------------
    # Prepare B
    # -----------------------------------------

    for i in range(8):
        if (b >> i) & 1:
            qc.x(8 + i)

    # -----------------------------------------
    # Prepare Mode
    # -----------------------------------------

    if mode == 1:
        qc.x(16)

    # -----------------------------------------
    # Initial carry/borrow
    # -----------------------------------------

    carry = 25

    # -----------------------------------------
    # Process each bit
    # -----------------------------------------

    for i in range(8):

        a_bit = i
        b_bit = 8 + i
        result_bit = 17 + i

        # Result = A XOR B XOR carry/borrow
        qc.cx(a_bit, result_bit)
        qc.cx(b_bit, result_bit)
        qc.cx(carry, result_bit)

        # Temporary A XOR Mode
        qc.cx(16, a_bit)

        # Carry/Borrow calculation
        qc.ccx(a_bit, b_bit, carry)

        # Restore A
        qc.cx(16, a_bit)

    return qc

In [3]:
qc = create_8bit_adder_subtractor(
    10,
    5,
    0
)

print(qc)

                                              ┌───┐                         »
 q_0: ───────■────────────────────────────────┤ X ├─────────────────────────»
      ┌───┐  │                                └─┬─┘                         »
 q_1: ┤ X ├──┼──────────────────────────────────┼────■──────────────────────»
      └───┘  │                                  │    │                      »
 q_2: ───────┼────■─────────────────────────────┼────┼──────────────────────»
      ┌───┐  │    │                             │    │                      »
 q_3: ┤ X ├──┼────┼─────────────────────────────┼────┼─────────■────────────»
      └───┘  │    │                             │    │         │            »
 q_4: ───────┼────┼────■────────────────────────┼────┼─────────┼────────────»
             │    │    │                        │    │         │            »
 q_5: ───────┼────┼────┼────■───────────────────┼────┼─────────┼────────────»
             │    │    │    │                   │    │         │

In [4]:
print("Number of qubits:", qc.num_qubits)
print("Circuit depth:", qc.depth())
print("Gate count:", qc.count_ops())

Number of qubits: 26
Circuit depth: 26
Gate count: OrderedDict({'cx': 40, 'ccx': 8, 'x': 4})


In [5]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator


def create_8bit_adder_subtractor(a, b, mode):
    qc = QuantumCircuit(34, 9)

    # Load A
    for i in range(8):
        if (a >> i) & 1:
            qc.x(i)

    # Load B
    for i in range(8):
        if (b >> i) & 1:
            qc.x(8 + i)

    # Mode
    if mode == 1:
        qc.x(16)

    # Initial carry = Mode
    qc.cx(16, 25)

    # 8-bit ripple stages
    for i in range(8):

        a_bit = i
        b_bit = 8 + i
        result_bit = 17 + i

        carry_in = 25 + i
        carry_out = 26 + i

        # B' = B XOR Mode
        qc.cx(16, b_bit)

        # Result = A XOR B' XOR Carry
        qc.cx(a_bit, result_bit)
        qc.cx(b_bit, result_bit)
        qc.cx(carry_in, result_bit)

        # Carry propagation
        qc.ccx(a_bit, b_bit, carry_out)
        qc.ccx(a_bit, carry_in, carry_out)
        qc.ccx(b_bit, carry_in, carry_out)

        # Restore B
        qc.cx(16, b_bit)

    # Measure result bits
    for i in range(8):
        qc.measure(17 + i, i)

    # Measure final carry
    qc.measure(33, 8)

    return qc

In [10]:
from qiskit_aer import AerSimulator

simulator = AerSimulator(
    method="matrix_product_state"
)

qc = create_8bit_adder_subtractor(10, 5, 0)

job = simulator.run(
    qc,
    shots=1
)

result = job.result()

print("Simulation completed.")

Simulation completed.


In [11]:
counts = result.get_counts()

print(counts)

{'000001111': 1}


In [14]:
# Test subtraction: 10 - 5

qc = create_8bit_adder_subtractor(10, 5, 1)

job = simulator.run(
    qc,
    shots=1
)

result = job.result()

counts = result.get_counts()

print(counts)

{'100000101': 1}


In [15]:
def decode_quantum_result(counts):
    bitstring = list(counts.keys())[0]

    # Last 8 bits are the result
    result_bits = bitstring[-8:]

    # First bit is final carry
    final_carry = int(bitstring[0])

    # Convert result binary to decimal
    result_value = int(result_bits, 2)

    return result_value, final_carry

In [16]:
result_value, final_carry = decode_quantum_result(counts)

print("Result:", result_value)
print("Final carry:", final_carry)

Result: 5
Final carry: 1


In [18]:
qc = create_8bit_adder_subtractor(5, 10, 1)

In [19]:
job = simulator.run(
    qc,
    shots=1
)

result = job.result()
counts = result.get_counts()

print(counts)

result_value, final_carry = decode_quantum_result(counts)

print("Result:", result_value)
print("Final carry:", final_carry)

{'011111011': 1}
Result: 251
Final carry: 0


In [20]:
def run_quantum_test(a, b, mode):
    """
    Run the 8-bit quantum adder-cum-subtractor
    and return the result and final carry.
    """

    qc = create_8bit_adder_subtractor(a, b, mode)

    job = simulator.run(
        qc,
        shots=1
    )

    result = job.result()
    counts = result.get_counts()

    result_value, final_carry = decode_quantum_result(counts)

    return result_value, final_carry

In [21]:
addition_tests = [
    (0, 0),
    (1, 1),
    (5, 10),
    (10, 5),
    (25, 30),
    (50, 75),
    (100, 50),
    (120, 100),
    (200, 30),
    (255, 0)
]

print("Addition Tests")
print("-" * 45)

for a, b in addition_tests:

    result, carry = run_quantum_test(a, b, 0)

    expected = a + b

    print(
        f"A={a:3d}, B={b:3d} | "
        f"Quantum={result:3d}, Carry={carry} | "
        f"Expected={expected}"
    )

Addition Tests
---------------------------------------------
A=  0, B=  0 | Quantum=  0, Carry=0 | Expected=0
A=  1, B=  1 | Quantum=  2, Carry=0 | Expected=2
A=  5, B= 10 | Quantum= 15, Carry=0 | Expected=15
A= 10, B=  5 | Quantum= 15, Carry=0 | Expected=15
A= 25, B= 30 | Quantum= 55, Carry=0 | Expected=55
A= 50, B= 75 | Quantum=125, Carry=0 | Expected=125
A=100, B= 50 | Quantum=150, Carry=0 | Expected=150
A=120, B=100 | Quantum=220, Carry=0 | Expected=220
A=200, B= 30 | Quantum=230, Carry=0 | Expected=230
A=255, B=  0 | Quantum=255, Carry=0 | Expected=255


In [22]:
subtraction_tests = [
    (10, 5),
    (20, 10),
    (50, 25),
    (100, 50),
    (200, 100),
    (255, 1),
    (5, 10),
    (25, 50),
    (50, 100),
    (100, 200)
]

print("Subtraction Tests")
print("-" * 50)

for a, b in subtraction_tests:

    result, final_carry = run_quantum_test(a, b, 1)

    if a >= b:
        expected = a - b
        borrow = 0
    else:
        expected = (a - b) % 256
        borrow = 1

    print(
        f"A={a:3d}, B={b:3d} | "
        f"Quantum={result:3d}, "
        f"Expected={expected:3d}, "
        f"Borrow={borrow}"
    )

Subtraction Tests
--------------------------------------------------
A= 10, B=  5 | Quantum=  5, Expected=  5, Borrow=0
A= 20, B= 10 | Quantum= 10, Expected= 10, Borrow=0
A= 50, B= 25 | Quantum= 25, Expected= 25, Borrow=0
A=100, B= 50 | Quantum= 50, Expected= 50, Borrow=0
A=200, B=100 | Quantum=100, Expected=100, Borrow=0
A=255, B=  1 | Quantum=254, Expected=254, Borrow=0
A=  5, B= 10 | Quantum=251, Expected=251, Borrow=1
A= 25, B= 50 | Quantum=231, Expected=231, Borrow=1
A= 50, B=100 | Quantum=206, Expected=206, Borrow=1
A=100, B=200 | Quantum=156, Expected=156, Borrow=1


In [23]:
verification_results = []

# Addition
for a, b in addition_tests:

    quantum_result, carry = run_quantum_test(a, b, 0)

    expected_result = (a + b) % 256
    expected_carry = 1 if (a + b) > 255 else 0

    passed = (
        quantum_result == expected_result
        and carry == expected_carry
    )

    verification_results.append({
        "A": a,
        "B": b,
        "Mode": "Addition",
        "Quantum_Result": quantum_result,
        "Expected_Result": expected_result,
        "Carry_Borrow": carry,
        "Expected_Carry_Borrow": expected_carry,
        "Status": "PASS" if passed else "FAIL"
    })


# Subtraction
for a, b in subtraction_tests:

    quantum_result, final_carry = run_quantum_test(a, b, 1)

    expected_result = (a - b) % 256
    expected_borrow = 1 if a < b else 0

    # In our circuit:
    # final_carry = 1 -> no borrow
    # final_carry = 0 -> borrow

    actual_borrow = 1 - final_carry

    passed = (
        quantum_result == expected_result
        and actual_borrow == expected_borrow
    )

    verification_results.append({
        "A": a,
        "B": b,
        "Mode": "Subtraction",
        "Quantum_Result": quantum_result,
        "Expected_Result": expected_result,
        "Carry_Borrow": actual_borrow,
        "Expected_Carry_Borrow": expected_borrow,
        "Status": "PASS" if passed else "FAIL"
    })

In [24]:
import pandas as pd

verification_df = pd.DataFrame(
    verification_results
)

display(verification_df)

,A,B,Mode,Quantum_Result,Expected_Result,Carry_Borrow,Expected_Carry_Borrow,Status
0,0,0,Addition,0,0,0,0,PASS
1,1,1,Addition,2,2,0,0,PASS
2,5,10,Addition,15,15,0,0,PASS
3,10,5,Addition,15,15,0,0,PASS
4,25,30,Addition,55,55,0,0,PASS
5,50,75,Addition,125,125,0,0,PASS
6,100,50,Addition,150,150,0,0,PASS
7,120,100,Addition,220,220,0,0,PASS
8,200,30,Addition,230,230,0,0,PASS
9,255,0,Addition,255,255,0,0,PASS


In [25]:
print(
    "Total tests:",
    len(verification_df)
)

print(
    "Passed:",
    (verification_df["Status"] == "PASS").sum()
)

print(
    "Failed:",
    (verification_df["Status"] == "FAIL").sum()
)

Total tests: 20
Passed: 20
Failed: 0


In [26]:
verification_df.to_csv(
    "../results/tables/8bit_quantum_verification.csv",
    index=False
)

print("Verification results saved.")

Verification results saved.
